# Créer un environnement Gymnasium personnalisé : Flappy Bird

## Question directrice

**Comment transformer les règles d'un petit jeu en environnement compatible avec Gymnasium, puis entraîner un agent à interagir avec lui ?**

Dans ce notebook, nous allons construire un environnement `FlappyBirdEnv` à partir de zéro. L'objectif n'est pas de reproduire le jeu commercial, mais de comprendre les contrats essentiels d'un environnement de Reinforcement Learning :

1. définir les espaces d'actions et d'observations ;
2. initialiser un épisode avec `reset()` ;
3. faire évoluer le monde avec `step(action)` ;
4. calculer les récompenses et les fins d'épisode ;
5. tester l'environnement avant d'entraîner un agent PPO.

Le jeu est volontairement simplifié : l'oiseau reste à une position horizontale fixe et les tuyaux se déplacent vers lui. Cette simplification permet de se concentrer sur la conception d'un environnement, plutôt que sur la reproduction graphique complète.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback

## 1. Préparer les bibliothèques

`gymnasium` fournit l'interface standard, `spaces` décrit les entrées et sorties possibles, et `numpy` sert à représenter les observations numériques. PPO et ses outils seront utilisés seulement après avoir validé l'environnement.

Une règle importante : avant d'entraîner un agent, on doit pouvoir tester l'environnement indépendamment de tout algorithme.

## 2. Le contrat d'un environnement Gymnasium

Un environnement personnalisé doit exposer quatre éléments :

- `action_space` : les actions que l'agent peut choisir ;
- `observation_space` : la forme et les bornes des observations ;
- `reset()` : démarre un nouvel épisode et retourne `(observation, info)` ;
- `step(action)` : applique une action et retourne `(observation, reward, terminated, truncated, info)`.

Dans Flappy Bird, l'action est discrète : `0` signifie ne pas battre des ailes et `1` signifie battre des ailes. L'observation sera un vecteur de cinq valeurs normalisées décrivant la position et la situation du prochain tuyau.

La cellule suivante implémente réellement ce contrat.

In [ ]:
# Permet de relancer la cellule de définition sans avertissement de doublon.
if "FlappyBird-v0" in gym.registry:
    del gym.registry["FlappyBird-v0"]

In [ ]:
class FlappyBirdEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}

    def __init__(self, render_mode=None):
        super().__init__()

        # Dimensions de l'écran (en "unités" internes, pas forcément pixels)
        self.screen_width = 400
        self.screen_height = 600

        # Physique (dt ; temps écoulé par image)
        self.dt = 1 / 30.0
        self.gravity = 900.0        # unités / s^2
        self.flap_strength = -320.0  # impulsion verticale négative = vers le haut en unités / s

        # Oiseau
        self.bird_x = 60.0          # position horizontale fixe
        self.bird_radius = 12.0

        # Tuyaux
        self.pipe_width = 50.0
        self.pipe_gap = 170.0       # taille du trou (fixe pour notre IA débutante on randomisera plus tard)
        self.pipe_speed = 180.0     # unités / s vers la gauche (c'est notre monde qui avance pas l'oiseau)
        self.pipe_spacing = 220.0   # distance horizontale entre 2 tuyaux

        # Limites d'épisode
        self.max_steps = 3000       # truncation de sécurité
        self.current_step = 0

        # Reward shaping
        self.reward_alive = 0.1
        self.reward_pipe = 5.0
        self.reward_death = -10.0

        # Action space : 0 = rien, 1 = flap
        self.action_space = spaces.Discrete(2)

        # Notre espace d'observation (5 valeurs normalisées)
        # bird_y_norm : Position de l'oiseau en Hauteur
        # bird_vy_norm : Vitesse verticale de l'oiseau
        # dist_x_norm : Distance Horizontale au prochain tuyau
        # dist_top_norm : Distance au haut du tuyau
        # dist_bottom_norm: Distance au bas du tuyau
        self.observation_space = spaces.Box(
            low=-1.0, high=1.0, shape=(5,), dtype=np.float32
        )

        self.render_mode = render_mode
        self._screen = None
        self._clock = None
        self._closed = False  # True une fois la fenêtre fermée (croix "X")

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # L'oiseau est à la moitié de l'ecran
        self.bird_y = self.screen_height / 2
        # Vitesse verticale initiale
        self.bird_vy = 0.0

        self.current_step = 0
        self.score = 0

        # On génère trois tuyaux en attent
        self.pipes = []
        start_x = self.screen_width + 100
        for i in range(3):
            self.pipes.append(self._make_pipe(start_x + i * self.pipe_spacing))

        obs = self._get_obs()
        info = {}
        return obs, info

    def _make_pipe(self, x):
        """Crée un tuyau avec une position de gap aléatoire (taille fixe)."""
        margin = 80  # marge pour ne pas coller le gap en haut/bas de l'écran
        gap_center = self.np_random.uniform(
            margin + self.pipe_gap / 2,
            self.screen_height - margin - self.pipe_gap / 2,
        )
        return {
            "x": x, # position horizontale du tuyau sur l'écran
            "gap_center": gap_center,
            "scored": False,
        }

    def step(self, action):
        self.current_step += 1

        # Physique de l'oiseau
        if action == 1:
            self.bird_vy = self.flap_strength
        self.bird_vy += self.gravity * self.dt
        self.bird_y += self.bird_vy * self.dt

        # Déplacement des tuyaux
        for pipe in self.pipes:
            pipe["x"] -= self.pipe_speed * self.dt

        # Recycle le tuyau sorti de l'écran à gauche
        if self.pipes[0]["x"] < -self.pipe_width:
            self.pipes.pop(0)
            last_x = self.pipes[-1]["x"]
            self.pipes.append(self._make_pipe(last_x + self.pipe_spacing))

        # Reward et collisions
        reward = self.reward_alive
        terminated = False

        # Score : un tuyau est franchi dès que son bord droit passe derrière l'oiseau.
        # On parcourt tous les tuyaux : _next_pipe() ignore ceux déjà dépassés.
        for pipe in self.pipes:
            if not pipe["scored"] and pipe["x"] + self.pipe_width < self.bird_x:
                pipe["scored"] = True
                self.score += 1
                reward += self.reward_pipe

        next_pipe = self._next_pipe()

        # Collision avec le sol ou le plafond
        if self.bird_y - self.bird_radius <= 0 or self.bird_y + self.bird_radius >= self.screen_height:
            terminated = True
            reward = self.reward_death

        # Collision avec un tuyau
        if self._check_pipe_collision(next_pipe):
            terminated = True
            reward = self.reward_death

        truncated = self.current_step >= self.max_steps

        obs = self._get_obs()
        info = {"score": self.score}

        return obs, reward, terminated, truncated, info

    def _next_pipe(self):
        """Retourne le prochain tuyau que l'oiseau doit franchir."""
        for pipe in self.pipes:
            if pipe["x"] + self.pipe_width >= self.bird_x:
                return pipe
        return self.pipes[-1]

    def _check_pipe_collision(self, pipe):
        bird_left = self.bird_x - self.bird_radius
        bird_right = self.bird_x + self.bird_radius
        pipe_left = pipe["x"]
        pipe_right = pipe["x"] + self.pipe_width

        # L'oiseau est dans la plage horizontale du tuyau ?
        if bird_right < pipe_left or bird_left > pipe_right:
            return False

        gap_top = pipe["gap_center"] - self.pipe_gap / 2
        gap_bottom = pipe["gap_center"] + self.pipe_gap / 2

        bird_top = self.bird_y - self.bird_radius
        bird_bottom = self.bird_y + self.bird_radius

        # Collision si l'oiseau dépasse le trou en haut ou en bas
        if bird_top < gap_top or bird_bottom > gap_bottom:
            return True
        return False

    def _get_obs(self):
        next_pipe = self._next_pipe()

        gap_top = next_pipe["gap_center"] - self.pipe_gap / 2
        gap_bottom = next_pipe["gap_center"] + self.pipe_gap / 2

        bird_y_norm = (self.bird_y / self.screen_height) * 2 - 1
        bird_vy_norm = np.clip(self.bird_vy / 600.0, -1, 1)
        dist_x_norm = np.clip(
            (next_pipe["x"] - self.bird_x) / self.screen_width, -1, 1
        )
        dist_top_norm = np.clip((self.bird_y - gap_top) / self.screen_height, -1, 1)
        dist_bottom_norm = np.clip((gap_bottom - self.bird_y) / self.screen_height, -1, 1)

        return np.array(
            [bird_y_norm, bird_vy_norm, dist_x_norm, dist_top_norm, dist_bottom_norm],
            dtype=np.float32,
        )

    def render(self):
        if self.render_mode is None:
            return

        import pygame

        if self._closed:
            return

        if self._screen is None:
            pygame.init()
            self._screen = pygame.display.set_mode((self.screen_width, self.screen_height))
            self._clock = pygame.time.Clock()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.close()
                return

        self._screen.fill((135, 206, 235))  # ciel bleu

        # Tuyaux
        for pipe in self.pipes:
            gap_top = pipe["gap_center"] - self.pipe_gap / 2
            gap_bottom = pipe["gap_center"] + self.pipe_gap / 2
            pygame.draw.rect(
                self._screen, (0, 150, 0),
                (pipe["x"], 0, self.pipe_width, gap_top),
            )
            pygame.draw.rect(
                self._screen, (0, 150, 0),
                (pipe["x"], gap_bottom, self.pipe_width, self.screen_height - gap_bottom),
            )

        # Oiseau
        pygame.draw.circle(
            self._screen, (255, 220, 0),
            (int(self.bird_x), int(self.bird_y)), int(self.bird_radius),
        )

        pygame.display.flip()
        self._clock.tick(self.metadata["render_fps"])

    def close(self):
        """Ferme proprement la fenêtre pygame (appelable plusieurs fois)."""
        self._closed = True
        if self._screen is not None:
            import pygame
            pygame.quit()
            self._screen = None


gym.register(
    id="FlappyBird-v0",
    entry_point=__name__+":FlappyBirdEnv",
    max_episode_steps=3000,
)

## 3. Concevoir la dynamique du jeu

La classe sépare volontairement plusieurs responsabilités :

- `reset()` place l'oiseau et crée trois tuyaux aléatoires ;
- `step()` applique la gravité, l'impulsion éventuelle et le déplacement des tuyaux ;
- `_check_pipe_collision()` détermine si l'oiseau touche un obstacle ;
- `_get_obs()` convertit l'état interne en vecteur normalisé ;
- `render()` fournit une visualisation indépendante de la logique de décision.

### Choix de conception

- **Terminaison** : une collision avec un tuyau, le plafond ou le sol met fin à l'épisode (`terminated=True`).
- **Troncature** : `max_steps` sert de limite de sécurité (`truncated=True`).
- **Récompense** : rester en vie rapporte `+0.1`, franchir un tuyau ajoute `+5`, et une collision donne `-10`.
- **Observation** : les cinq valeurs sont bornées dans `[-1, 1]`, ce qui facilite l'apprentissage d'un réseau de neurones.

Cette séparation est importante : l'agent ne voit pas les variables internes comme `bird_y` ou `pipes`, il ne reçoit que l'observation définie par l'API.

## 4. Premier test : vérifier le contrat, pas encore l'apprentissage

Avant PPO, on vérifie que l'environnement respecte les attentes de Gymnasium. Ce test contrôle la forme de l'observation, la validité des actions et la structure de la sortie de `step()`.

Le test utilise `render_mode=None` : l'affichage sera testé séparément, car une fenêtre graphique ne doit pas être nécessaire pour vérifier la logique du jeu.

In [ ]:
env = gym.make("FlappyBird-v0")
obs, info = env.reset(seed=42)

assert env.observation_space.contains(obs)
assert env.action_space.n == 2

for step in range(200):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    assert env.observation_space.contains(obs)
    assert isinstance(reward, (int, float, np.integer, np.floating))
    assert isinstance(terminated, bool)
    assert isinstance(truncated, bool)
    assert "score" in info

    if terminated or truncated:
        obs, info = env.reset(seed=42 + step)

env.close()
print("Smoke test réussi : reset(), step(), observations et signaux de fin sont valides.")

## 5. Visualiser l'environnement

Le rendu n'est pas nécessaire à l'apprentissage : PPO utilise les observations numériques. Il est cependant précieux pour comprendre le comportement appris et repérer une incohérence entre la logique et l'affichage.

La méthode `render()` utilise Pygame uniquement lorsque `render_mode="human"`. Pour un test rapide, on peut donc lancer une politique aléatoire pendant quelques épisodes, puis fermer explicitement l'environnement avec `close()`.

## 6. Préparer l'entraînement PPO

PPO est adapté ici parce que l'observation est un vecteur continu de cinq valeurs et que l'action reste discrète. `MlpPolicy` apprend une politique à partir de ce vecteur, sans image et sans accès direct aux variables internes du jeu.

### Pourquoi vectoriser les environnements ?

Avec `make_vec_env`, huit épisodes peuvent progresser en parallèle. Cela augmente la quantité d'expérience collectée par unité de temps et rend l'entraînement plus stable. L'environnement d'évaluation reste séparé et non vectorisé afin de mesurer la politique dans des conditions lisibles.

### Rôle du callback

`EvalCallback` évalue périodiquement l'agent et sauvegarde le meilleur modèle. L'entraînement et l'évaluation doivent utiliser des environnements distincts pour éviter de mélanger les données d'apprentissage et les résultats de test.

In [ ]:
# Huit environnements d'entraînement exécutés en parallèle.
train_env = make_vec_env(FlappyBirdEnv, n_envs=8)
eval_env = FlappyBirdEnv()

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="models/",
    log_path="logs/",
    eval_freq=10_000,
    deterministic=True,
    render=False,
)

model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    tensorboard_log="logs/tb_logs/",
)

In [ ]:
# Entraînement long : cette cellule peut prendre du temps.
model.learn(total_timesteps=1_000_000, callback=eval_callback)

model.save("models/flappy_ppo_final")
train_env.close()
eval_env.close()

print("Entraînement terminé et modèle sauvegardé dans models/flappy_ppo_final.zip")

## 7. Évaluer l'agent sur plusieurs épisodes

Une seule partie ne suffit pas pour conclure. Nous évaluons la politique sur 20 épisodes indépendants, sans exploration aléatoire. Un agent bien entraîné ne tombe plus : chaque épisode va alors jusqu'à la limite de 3000 pas (100 s de jeu), d'où un nombre d'épisodes réduit. On mesure :

- la récompense totale moyenne et sa dispersion ;
- le score moyen, qui compte les tuyaux franchis ;
- la longueur moyenne des épisodes ;
- le taux d'épisodes qui franchissent au moins un tuyau ;
- le taux de survie : la part d'épisodes qui atteignent la limite de 3000 pas sans collision.

L'évaluation est sans rendu graphique afin de mesurer la politique sans ralentir la collecte.

In [ ]:
evaluation_env = FlappyBirdEnv()
evaluation_model = PPO.load("models/flappy_ppo_final")

n_eval_episodes = 20  # chaque épisode peut durer 3000 pas : 20 suffisent pour un agent qui ne tombe plus
episode_rewards = []
episode_scores = []
episode_lengths = []
episode_survived = []

for episode in range(n_eval_episodes):
    obs, info = evaluation_env.reset(seed=episode)
    total_reward = 0.0
    steps = 0
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action, _ = evaluation_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = evaluation_env.step(int(action))
        total_reward += reward
        steps += 1

    episode_rewards.append(total_reward)
    episode_scores.append(info["score"])
    episode_lengths.append(steps)
    episode_survived.append(truncated and not terminated)

evaluation_env.close()

rewards = np.asarray(episode_rewards, dtype=np.float32)
scores = np.asarray(episode_scores, dtype=np.int32)
survived = np.asarray(episode_survived, dtype=bool)
lengths = np.asarray(episode_lengths, dtype=np.int32)

print(f"=== Évaluation PPO sur {n_eval_episodes} épisodes ===")
print(f"Récompense moyenne : {rewards.mean():.2f}")
print(f"Écart-type         : {rewards.std():.2f}")
print(f"Récompense min/max : {rewards.min():.2f} / {rewards.max():.2f}")
print(f"Score moyen        : {scores.mean():.2f}")
print(f"Score médian       : {np.median(scores):.2f}")
print(f"Pas moyens         : {lengths.mean():.2f}")
print(f"Taux de réussite   : {np.mean(scores > 0):.1%}")
print(f"Taux de survie     : {survived.mean():.1%}  (épisodes allés jusqu'à {evaluation_env.max_steps} pas)")

## 8. Observer la politique apprise

La dernière cellule recharge le modèle sauvegardé et le fait jouer avec `deterministic=True`. Cette démonstration est qualitative : les statistiques précédentes restent la référence pour comparer les performances.

In [ ]:
env = FlappyBirdEnv(render_mode="human")
model = PPO.load("models/flappy_ppo_final")

obs, info = env.reset()

try:
    while not env._closed:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))
        env.render()

        if terminated or truncated:
            print(f"Score : {info['score']}")
            obs, info = env.reset()
finally:
    env.close()

## À retenir

Créer un environnement personnalisé revient à définir une interface claire entre le monde et l'agent. La qualité de cette interface dépend de trois choix : une observation suffisamment informative, une récompense cohérente avec l'objectif et des signaux `terminated`/`truncated` correctement séparés.

Une fois ces contrats testés indépendamment, PPO peut traiter l'environnement comme n'importe quel environnement Gymnasium standard.